# Polynomial Regression (Simple Example)

Fit a curved relationship by turning one feature into powers and using **linear** regression on the result.

A degree-`d` polynomial predicts:

$$\hat{y} = \beta_0 + \beta_1 x + \beta_2 x^{2} + \dots + \beta_d x^{d}$$

We build a **design matrix** with columns $[1, x, x^{2}, \dots, x^{d}]$ and solve the Normal Equation:

$$\hat{\boldsymbol{\beta}} = (\mathbf{X}^{\top}\mathbf{X})^{-1}\,\mathbf{X}^{\top}\mathbf{y}$$


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Dataset: 6 points following y = 1 + 2x + 0.5 x^2
x = np.array([1, 2, 3, 4, 5, 6], dtype=float)
y = 1 + 2 * x + 0.5 * x ** 2

df = pd.DataFrame({"x": x, "y": y})
print(df.to_string(index=False))


  x    y
1.0  3.5
2.0  7.0
3.0 11.5
4.0 17.0
5.0 23.5
6.0 31.0


## Build the polynomial design matrix and fit

`make_design_matrix` returns columns $[1, x, x^{2}, \dots, x^{d}]$, then we apply the Normal Equation.


In [ ]:
def make_design_matrix(x, degree):
    cols = [np.ones_like(x)]
    for p in range(1, degree + 1):
        cols.append(x ** p)
    return np.column_stack(cols)

DEGREE = 2
X_design = make_design_matrix(x, DEGREE)
print("=== DESIGN MATRIX ===")
print(pd.DataFrame(X_design, columns=[f"x^{p}" for p in range(DEGREE + 1)]).to_string(index=False))

beta = np.linalg.inv(X_design.T @ X_design) @ X_design.T @ y

print("=== LEARNED PARAMETERS ===")
labels = ["Intercept"] + [f"x^{p}" for p in range(1, DEGREE + 1)]
for lab, b in zip(labels, beta):
    print(f"{lab:<10} : {b:.4f}")


=== LEARNED PARAMETERS ===
Intercept  : 1.0000
x^1        : 2.0000
x^2        : 0.5000


## Performance Metrics & Residual Analysis


In [ ]:
y_pred = X_design @ beta
residuals = y - y_pred

mse = np.mean(residuals ** 2)
rmse = np.sqrt(mse)
ss_res = np.sum(residuals ** 2)
ss_tot = np.sum((y - np.mean(y)) ** 2)
r2 = 1 - ss_res / ss_tot

print("=== METRICS ===")
print(f"Mean Squared Error (MSE) : {mse:.4f}")
print(f"Root MSE (RMSE)          : {rmse:.4f}")
print(f"R-squared (R2)           : {r2:.4f}")

res_df = pd.DataFrame({
    "x": x,
    "y actual": y,
    "y predicted": np.round(y_pred, 3),
    "residual": np.round(residuals, 3),
})
print()
print("=== PER-POINT RESIDUALS ===")
print(res_df.to_string(index=False))


## Plot


In [ ]:
x_line = np.linspace(x.min() - 0.5, x.max() + 0.5, 200)
X_line = make_design_matrix(x_line, DEGREE)
y_line = X_line @ beta

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Fit curve
axes[0].scatter(x, y, color="red", label="Actual", zorder=5)
axes[0].plot(x_line, y_line, color="blue", label=f"Polynomial (deg {DEGREE})")
axes[0].set_title("Polynomial Fit")
axes[0].set_xlabel("x")
axes[0].set_ylabel("y")
axes[0].legend()
axes[0].grid(True, linestyle="--", alpha=0.4)

# Residuals
axes[1].axhline(0, color="red", linestyle="--", lw=1.5)
axes[1].scatter(x, residuals, color="green", s=80, edgecolor="black", zorder=5)
axes[1].set_title("Residuals")
axes[1].set_xlabel("x")
axes[1].set_ylabel("Residual")
axes[1].grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()
